# Knee range of motion from pose landmarks: Markdown alone versus a scripted skill

Project 2 (rehab app) analytical task: screen a patient's knee range of motion
(ROM) during a seated knee extension from side-view 2D pose landmarks (hip,
knee, ankle), and decide **on_track**, **below_target**, or **clinician_review**.

Two versions of the same expert procedure are compared:
1. `knee-rom-markdown`: a self-contained `SKILL.md` with the geometry, frame-quality
   rules, uncertainty, decision rules, verification, and review triggers. The agent
   may load the skill and read the session file but cannot execute anything.
2. `knee-rom-scripted`: the same procedure plus bundled Python scripts
   (`joint_angles.py`, `analyze_knee_rom.py`) that the agent runs directly.
   The tool contract (typed inputs, valid ranges, ValueError on invalid input,
   structured results, warnings) is adapted from my HW5 car-setup MCP tools.

Neither condition uses MCP. Same model, same task prompt (apart from the skill
name), same input file, fresh session for each run. The session data is
synthetic: landmarks were generated from known geometry, with one out-of-plane
frame and one low-visibility frame planted to test frame rejection. A correct
answer does not prove skill use: inspect the skill event, the script command and
its output. This is a classroom screen, not a clinical tool.

## 1. Start a T4 GPU runtime
Choose **Runtime → Change runtime type → T4 GPU**, then run the setup cells through
**Model ready**. Run the two case cells separately. No inference API key is needed.

In [ ]:
%pip -q install "transformers[serving]==5.22.2"
!npm install -g @opencode/cli@2.2.6

import json, os, subprocess, time
from pathlib import Path

MODEL_ID = 'Qwen/Qwen3-4B'
ROOT = Path('/content/knee-rom-skills-demo')
MARKDOWN_RUN = ROOT / 'markdown-only'
ASSISTED_RUN = ROOT / 'script-assisted'
TRANSCRIPTS = ROOT / 'transcripts'
TRANSCRIPTS.mkdir(parents=True, exist_ok=True)

## 2. Prepare the two skills, the session input, and the agents
Case 1 contains only `SKILL.md`. Case 2 adds `scripts/` and `references/`.
Both run folders receive an identical copy of `session_input.json`.

In [ ]:
#@title Prepare both skills, the input file, and agents
SKILL_FILES = {
 "skills/knee-rom-markdown/SKILL.md": "---\nname: knee-rom-markdown\ndescription: Screen knee range of motion for a rehab patient from side-view 2D pose landmarks (hip, knee, ankle) during a seated knee extension. Compute knee flexion per frame by hand, reject unreliable frames, summarize extension lag and ROM, compare with clinician targets and the last session, and decide on_track, below_target, or clinician_review. Use for rehab joint-angle evaluation questions.\n---\n\nThis is the Markdown-only condition. This directory contains only SKILL.md.\nWork through the procedure below by hand and show the arithmetic for every\nframe (vectors, dot product, lengths, cosine, angle, flexion). Do not call MCP\nservices, shell commands, calculators, Python, or other computational tools.\nNative skill loading and reading the session input file are allowed.\nIf a calculation cannot be completed confidently, report the uncertainty and\nthe frames affected instead of inventing a value. Do not claim an executed or\nindependent numerical check. Arccos values computed by hand are approximate:\nstate them to the nearest 0.5° and say they are hand estimates.\n\n# Expert procedure: knee range-of-motion screen from 2D pose landmarks\n\n## Scope and analysis selection\n\nUse this procedure for ONE knee during ONE exercise recorded from the side\n(sagittal view) by a single phone camera, where a pose model (for example\nMediaPipe BlazePose) supplies hip, knee, and ankle landmarks per frame. The\nreference exercise is a seated knee extension: the patient sits, thigh\nroughly horizontal, and straightens the knee from about 90° of flexion toward\n0°. The output is a screening summary for a clinician, not a diagnosis, a\nprescription, or permission to progress the exercise plan.\n\nConfirm before calculating:\n- camera_view is \"sagittal\". A frontal or oblique view projects the knee\n  angle onto the wrong plane; 2D flexion from those views is not valid. Do not\n  \"correct\" it; stop and route to clinician review.\n- One side is analyzed (left or right), and all three landmarks belong to that\n  side. Mixing a left hip with a right knee produces a meaningless angle.\n- The exercise matches. Squats, gait, and stair climbing involve trunk and\n  hip motion and other targets; do not reuse these targets for them.\n\n## Inputs, units, and coordinate meaning\n\nRequired: side, camera_view, coordinate_units, frames (each with time t_s and\nhip/knee/ankle as [x, y, visibility]), clinician targets\n(max_extension_lag_deg, min_rom_deg), and patient-reported pain_0_10.\nOptional: previous_session (extension_lag_deg, rom_deg, pain_0_10).\nIf targets or pain are missing, do not invent them: ask for them, and report\nonly the measured angles, with no on_track / below_target decision.\n\nUnits: landmark x, y in pixels with the image origin at top-left and y\npointing DOWN; angles in degrees; time in seconds; pain on a 0–10 scale.\nIf coordinates are normalized (0–1, as MediaPipe outputs), multiply x by\nimage width and y by image height BEFORE computing any angle. Normalized x and\ny have different scales on a non-square (e.g. 1080×1920 portrait) image, so\nangles computed on unscaled values are distorted. Without image size, stop.\n\nVisibility is the pose model's confidence that a landmark is visible and\ncorrectly placed. It is not a calibrated probability, and a high score does\nnot guarantee a correct landmark.\n\n## Joint-angle geometry\n\nFor each frame, let H = hip, K = knee, A = ankle (pixel coordinates).\n1. Thigh vector u = H − K = (Hx − Kx, Hy − Ky). Shank vector v = A − K.\n2. Dot product u·v = ux·vx + uy·vy. Lengths |u| = √(ux² + uy²), |v| = √(vx² + vy²).\n3. Included angle θ = arccos( u·v / (|u||v|) ), between 0° and 180°. Clamp the\n   cosine to [−1, 1] before arccos to absorb rounding.\n4. Clinical knee flexion = 180° − θ. A straight leg (H, K, A collinear) has\n   θ = 180° and flexion 0°. A right angle at the knee is flexion 90°.\n\nSanity examples: H=(100,500), K=(500,500), A=(500,900) gives u=(−400,0),\nv=(0,400), u·v=0, θ=90°, flexion 90°. H=(100,500), K=(500,500), A=(900,500)\ngives θ=180°, flexion 0°. Reporting θ itself as \"flexion\" is the most common\nerror; a seated patient with a nearly straight leg must read near 0°, not 180°.\n\nThe unsigned arccos angle cannot represent hyperextension (knee straighter\nthan 0°). Values near 0° may hide hyperextension; say so instead of reporting\na negative number.\n\n## Frame quality: which frames to trust\n\nReject a frame, and keep the reason, when:\n- any of hip, knee, ankle has visibility < 0.5; or\n- the thigh length |u| or shank length |v| differs from its median across the\n  visibility-passing frames by more than 15%.\nIn a true side view, thigh and shank pixel lengths stay nearly constant.\nA segment that suddenly shortens means the leg rotated out of the camera plane\nor a landmark jumped; the 2D angle in that frame is wrong, and it often makes\nthe knee look straighter or more bent than it really is. Never keep such a\nframe because its angle looks favorable.\n\nCompute medians from the visibility-passing frames, not from all frames.\nIf fewer than 5 frames remain, or fewer than 60% of frames remain, the session\ncannot support a decision: report clinician_review.\nFor long videos (15 or more valid frames), apply a centered median filter of\nwidth 5 to the flexion series before taking extremes; with fewer frames, do not\nsmooth, because a filter would erase the real end-range frames.\n\n## Session metrics\n\nFrom the valid (and, if applicable, smoothed) flexion series:\n- extension lag = minimum flexion (how far short of straight the knee stays);\n- peak flexion = maximum flexion;\n- ROM = peak flexion − extension lag.\nKeep unrounded values through the comparisons; round to 0.1° only for display.\nThese are session extremes; this procedure does not segment repetitions.\n\n## Measurement uncertainty\n\nSingle-camera pose estimation is not goniometry. One knee-gait study reported\na mean absolute error of 5.88° for MediaPipe knee angles against Kinovea\n(IOP Conf. Ser.: Mater. Sci. Eng. 1279, 012004, doi:10.1088/1757-899x/1279/1/012004).\nA 2026 JMIR Rehabilitation and Assistive Technologies study (Yue, Gu, Yu,\ne102399) found static BlazePose accuracy depended on flexion angle, with RMSE\nabout 15° against a goniometer, overestimating at shallow flexion and\nunderestimating at deep flexion. This procedure therefore uses a measurement\ntolerance of 5° (a team-chosen screening default, not a validated minimal\ndetectable change). A difference smaller than the tolerance is not evidence of\nchange, and a target missed by less than the tolerance is \"within tolerance\",\nnot met.\n\n## Decision rules\n\nTargets come from the clinician for this patient; normative values are only\ncontext. Commonly cited normal knee motion is about 0° extension to 135°\nflexion (AAOS reference values; see also PMC3435893, which stresses comparing\nwith the patient's other knee). Rising from a chair needs roughly 90° flexion.\nDo not substitute these numbers for the clinician's targets.\n\nFor each target compute the miss:\n- extension lag miss = extension lag − max_extension_lag_deg;\n- ROM miss = min_rom_deg − ROM.\nStatus: miss ≤ 0 → met; 0 < miss ≤ tolerance → within_tolerance; miss >\ntolerance → not_met.\n\nChange versus previous session (improvement positive):\n- extension: previous lag − current lag; ROM: current ROM − previous ROM.\nClass: > +tolerance improved; < −tolerance worsened; otherwise\nno_detectable_change.\n\nClinician-review triggers (any one forces decision = clinician_review):\n- camera view not sagittal;\n- insufficient reliable frames (rule above);\n- implausible peak flexion > 150° for this exercise;\n- either metric worsened beyond the tolerance versus last session;\n- pain ≥ 6/10, or pain increased by ≥ 2 points since last session.\n\nIf no trigger fires: decision = below_target when any target is not_met;\notherwise on_track. If a target is within_tolerance, add the note \"confirm\nwith a goniometer before calling it met\". All thresholds here (0.5, 15%, 5°,\n150°, pain 6 and +2) are team-chosen screening defaults for a course project,\nnot clinical standards; a clinician must approve them before real use.\n\n## Verification, reporting, and human review\n\nCheck before reporting:\n- the flexion convention (straight leg ≈ 0°) on at least one frame;\n- that every rejected frame has a stated reason, and the valid count;\n- that ROM = peak flexion − extension lag;\n- that conclusions about change respect the tolerance;\n- one frame recomputed independently (by hand from the coordinates, or a\n  goniometer reading on the patient) and labeled with its source.\n\nReport a compact table: per frame t_s, flexion, thigh and shank length,\nstatus and reason; then extension lag, peak flexion, ROM, each target with\nstatus, change versus last session with class, pain, review triggers, and the\ndecision. Explain the decision in plain language: which number drove it and\nwhich frames were excluded.\n\nA person must review the output whenever: any review trigger fires; data or\ntargets are missing; a target is within tolerance; the result will change the\nexercise plan; or the patient reports new symptoms (swelling, instability,\nlocking, sharp pain). The screen omits swelling, strength, quadriceps lag\nversus passive extension, compensation at the hip or trunk, pain quality,\nsurgical precautions, and 3D motion. Name any of these that could change the\ninterpretation before giving the result.\n",
 "skills/knee-rom-scripted/SKILL.md": "---\nname: knee-rom-scripted\ndescription: Screen knee range of motion for a rehab patient from side-view 2D pose landmarks (hip, knee, ankle) during a seated knee extension, using bundled deterministic Python scripts. Compute knee flexion per frame, reject unreliable frames, summarize extension lag and ROM, compare with clinician targets and the last session, decide on_track, below_target, or clinician_review, then interpret and verify the result.\n---\n\nThis is the script-assisted condition. Read the procedure below, check that\nthe case is in scope (side view, one knee, seated knee extension, targets and\npain present), then run scripts/analyze_knee_rom.py on the session JSON file.\nThere is no MCP server, custom tool registration, or hosted API. Resolve the\nscript path relative to this skill's directory, not the shell's working\ndirectory; use the absolute path if they differ. Pass the session file by its\nabsolute path.\n\nCommand (replace the input path with the user's actual file):\n\n```bash\npython /content/knee-rom-skills-demo/script-assisted/.agents/skills/knee-rom-scripted/scripts/analyze_knee_rom.py \\\n  --input /content/knee-rom-skills-demo/script-assisted/session_input.json\n```\n\nRun it once with default settings. Override --visibility-min,\n--segment-tolerance, or --measurement-tolerance-deg only if the user or\nclinician explicitly supplies different values; never loosen a threshold to\nmake a session pass. The script imports joint_angles.py from its own folder\nand prints JSON with per-frame angles, segment lengths, frame status and\nrejection reasons, the session summary, target status, change versus the last\nsession, pain, review triggers, warnings, assumptions, and the decision.\n\nHow to use the output:\n- Preserve the actual command and the full JSON output in the transcript.\n- Interpret it; do not just repeat it. Say which frames were rejected and why,\n  which number drove the decision, and what \"within_tolerance\" or\n  \"no_detectable_change\" means for this patient.\n- Exit code 2 prints {\"error\": ...} on stderr: the input was invalid or\n  incomplete. Report the error message, ask for the missing or corrected\n  input, and stop. Do not fix the input yourself by guessing values.\n- Any other failure (non-JSON output, a missing field, a value that\n  contradicts the procedure, such as a straight leg reading near 180°): report\n  it as a failure, keep the discrepancy, and request review. Do not\n  manufacture output and do not fall back to hand-computing a decision.\n- The script is the primary computation, not an independent check. Recompute\n  the flexion of at least one valid frame by hand from its coordinates using\n  the geometry section below, compare with the script, and label the source of\n  that check. Agreement between two copies of the same function is not\n  independence.\n- references/REFERENCES.md lists the sources behind the uncertainty and\n  normative values; read it if the user asks where a number comes from.\n\n# Expert procedure: knee range-of-motion screen from 2D pose landmarks\n\n## Scope and analysis selection\n\nUse this procedure for ONE knee during ONE exercise recorded from the side\n(sagittal view) by a single phone camera, where a pose model (for example\nMediaPipe BlazePose) supplies hip, knee, and ankle landmarks per frame. The\nreference exercise is a seated knee extension: the patient sits, thigh\nroughly horizontal, and straightens the knee from about 90° of flexion toward\n0°. The output is a screening summary for a clinician, not a diagnosis, a\nprescription, or permission to progress the exercise plan.\n\nConfirm before calculating:\n- camera_view is \"sagittal\". A frontal or oblique view projects the knee\n  angle onto the wrong plane; 2D flexion from those views is not valid. Do not\n  \"correct\" it; stop and route to clinician review.\n- One side is analyzed (left or right), and all three landmarks belong to that\n  side. Mixing a left hip with a right knee produces a meaningless angle.\n- The exercise matches. Squats, gait, and stair climbing involve trunk and\n  hip motion and other targets; do not reuse these targets for them.\n\n## Inputs, units, and coordinate meaning\n\nRequired: side, camera_view, coordinate_units, frames (each with time t_s and\nhip/knee/ankle as [x, y, visibility]), clinician targets\n(max_extension_lag_deg, min_rom_deg), and patient-reported pain_0_10.\nOptional: previous_session (extension_lag_deg, rom_deg, pain_0_10).\nIf targets or pain are missing, do not invent them: ask for them, and report\nonly the measured angles, with no on_track / below_target decision.\n\nUnits: landmark x, y in pixels with the image origin at top-left and y\npointing DOWN; angles in degrees; time in seconds; pain on a 0–10 scale.\nIf coordinates are normalized (0–1, as MediaPipe outputs), multiply x by\nimage width and y by image height BEFORE computing any angle. Normalized x and\ny have different scales on a non-square (e.g. 1080×1920 portrait) image, so\nangles computed on unscaled values are distorted. Without image size, stop.\n\nVisibility is the pose model's confidence that a landmark is visible and\ncorrectly placed. It is not a calibrated probability, and a high score does\nnot guarantee a correct landmark.\n\n## Joint-angle geometry\n\nFor each frame, let H = hip, K = knee, A = ankle (pixel coordinates).\n1. Thigh vector u = H − K = (Hx − Kx, Hy − Ky). Shank vector v = A − K.\n2. Dot product u·v = ux·vx + uy·vy. Lengths |u| = √(ux² + uy²), |v| = √(vx² + vy²).\n3. Included angle θ = arccos( u·v / (|u||v|) ), between 0° and 180°. Clamp the\n   cosine to [−1, 1] before arccos to absorb rounding.\n4. Clinical knee flexion = 180° − θ. A straight leg (H, K, A collinear) has\n   θ = 180° and flexion 0°. A right angle at the knee is flexion 90°.\n\nSanity examples: H=(100,500), K=(500,500), A=(500,900) gives u=(−400,0),\nv=(0,400), u·v=0, θ=90°, flexion 90°. H=(100,500), K=(500,500), A=(900,500)\ngives θ=180°, flexion 0°. Reporting θ itself as \"flexion\" is the most common\nerror; a seated patient with a nearly straight leg must read near 0°, not 180°.\n\nThe unsigned arccos angle cannot represent hyperextension (knee straighter\nthan 0°). Values near 0° may hide hyperextension; say so instead of reporting\na negative number.\n\n## Frame quality: which frames to trust\n\nReject a frame, and keep the reason, when:\n- any of hip, knee, ankle has visibility < 0.5; or\n- the thigh length |u| or shank length |v| differs from its median across the\n  visibility-passing frames by more than 15%.\nIn a true side view, thigh and shank pixel lengths stay nearly constant.\nA segment that suddenly shortens means the leg rotated out of the camera plane\nor a landmark jumped; the 2D angle in that frame is wrong, and it often makes\nthe knee look straighter or more bent than it really is. Never keep such a\nframe because its angle looks favorable.\n\nCompute medians from the visibility-passing frames, not from all frames.\nIf fewer than 5 frames remain, or fewer than 60% of frames remain, the session\ncannot support a decision: report clinician_review.\nFor long videos (15 or more valid frames), apply a centered median filter of\nwidth 5 to the flexion series before taking extremes; with fewer frames, do not\nsmooth, because a filter would erase the real end-range frames.\n\n## Session metrics\n\nFrom the valid (and, if applicable, smoothed) flexion series:\n- extension lag = minimum flexion (how far short of straight the knee stays);\n- peak flexion = maximum flexion;\n- ROM = peak flexion − extension lag.\nKeep unrounded values through the comparisons; round to 0.1° only for display.\nThese are session extremes; this procedure does not segment repetitions.\n\n## Measurement uncertainty\n\nSingle-camera pose estimation is not goniometry. One knee-gait study reported\na mean absolute error of 5.88° for MediaPipe knee angles against Kinovea\n(IOP Conf. Ser.: Mater. Sci. Eng. 1279, 012004, doi:10.1088/1757-899x/1279/1/012004).\nA 2026 JMIR Rehabilitation and Assistive Technologies study (Yue, Gu, Yu,\ne102399) found static BlazePose accuracy depended on flexion angle, with RMSE\nabout 15° against a goniometer, overestimating at shallow flexion and\nunderestimating at deep flexion. This procedure therefore uses a measurement\ntolerance of 5° (a team-chosen screening default, not a validated minimal\ndetectable change). A difference smaller than the tolerance is not evidence of\nchange, and a target missed by less than the tolerance is \"within tolerance\",\nnot met.\n\n## Decision rules\n\nTargets come from the clinician for this patient; normative values are only\ncontext. Commonly cited normal knee motion is about 0° extension to 135°\nflexion (AAOS reference values; see also PMC3435893, which stresses comparing\nwith the patient's other knee). Rising from a chair needs roughly 90° flexion.\nDo not substitute these numbers for the clinician's targets.\n\nFor each target compute the miss:\n- extension lag miss = extension lag − max_extension_lag_deg;\n- ROM miss = min_rom_deg − ROM.\nStatus: miss ≤ 0 → met; 0 < miss ≤ tolerance → within_tolerance; miss >\ntolerance → not_met.\n\nChange versus previous session (improvement positive):\n- extension: previous lag − current lag; ROM: current ROM − previous ROM.\nClass: > +tolerance improved; < −tolerance worsened; otherwise\nno_detectable_change.\n\nClinician-review triggers (any one forces decision = clinician_review):\n- camera view not sagittal;\n- insufficient reliable frames (rule above);\n- implausible peak flexion > 150° for this exercise;\n- either metric worsened beyond the tolerance versus last session;\n- pain ≥ 6/10, or pain increased by ≥ 2 points since last session.\n\nIf no trigger fires: decision = below_target when any target is not_met;\notherwise on_track. If a target is within_tolerance, add the note \"confirm\nwith a goniometer before calling it met\". All thresholds here (0.5, 15%, 5°,\n150°, pain 6 and +2) are team-chosen screening defaults for a course project,\nnot clinical standards; a clinician must approve them before real use.\n\n## Verification, reporting, and human review\n\nCheck before reporting:\n- the flexion convention (straight leg ≈ 0°) on at least one frame;\n- that every rejected frame has a stated reason, and the valid count;\n- that ROM = peak flexion − extension lag;\n- that conclusions about change respect the tolerance;\n- one frame recomputed independently (by hand from the coordinates, or a\n  goniometer reading on the patient) and labeled with its source.\n\nReport a compact table: per frame t_s, flexion, thigh and shank length,\nstatus and reason; then extension lag, peak flexion, ROM, each target with\nstatus, change versus last session with class, pain, review triggers, and the\ndecision. Explain the decision in plain language: which number drove it and\nwhich frames were excluded.\n\nA person must review the output whenever: any review trigger fires; data or\ntargets are missing; a target is within tolerance; the result will change the\nexercise plan; or the patient reports new symptoms (swelling, instability,\nlocking, sharp pain). The screen omits swelling, strength, quadriceps lag\nversus passive extension, compensation at the hip or trunk, pain quality,\nsurgical precautions, and 3D motion. Name any of these that could change the\ninterpretation before giving the result.\n",
 "skills/knee-rom-scripted/references/REFERENCES.md": "# Sources behind numbers in the knee ROM skill\n\n- Measurement error, gait, MediaPipe vs Kinovea: mean absolute knee-angle error\n  5.88°, correlation 0.941. \"Knee Flexion/Extension Angle Measurement for Gait\n  Analysis Using Machine Learning Solution 'MediaPipe Pose' and Its Comparison\n  with Kinovea\", IOP Conf. Ser.: Mater. Sci. Eng. 1279, 012004,\n  doi:10.1088/1757-899x/1279/1/012004.\n- Static accuracy, BlazePose vs goniometer at 30/60/90/120°: angle-dependent,\n  RMSE about 15.5°, overestimates shallow and underestimates deep flexion.\n  Included hip-knee-ankle angles converted to clinical flexion (0° = full\n  extension). Yue XA, Gu Y, Yu B. JMIR Rehabil Assist Technol 2026;13:e102399.\n- Normal knee motion and the need to compare with the other knee:\n  \"What is 'Normal' Knee ROM?\", PMC3435893 (0° extension, 135° flexion as a\n  common definition; many people have hyperextension and more flexion).\n- Normative ROM dataset: Soucie JM et al. 2010, CDC public-use dataset for\n  normal joint range of motion (674 subjects), stacks.cdc.gov/view/cdc/153156.\n\nThresholds in SKILL.md (visibility 0.5, segment 15%, tolerance 5°, flexion\ncap 150°, pain 6/10 or +2) are team-chosen screening defaults informed by the\nerror magnitudes above. They are not validated clinical cut-offs.\n",
 "skills/knee-rom-scripted/scripts/analyze_knee_rom.py": "\"\"\"Run the knee ROM screen on a session JSON file and print structured JSON.\n\nUsage:\n  python analyze_knee_rom.py --input session_input.json [--visibility-min 0.5]\n      [--segment-tolerance 0.15] [--measurement-tolerance-deg 5]\n\nExit codes: 0 success (JSON on stdout); 2 invalid input or failed calculation\n(JSON {\"error\": ...} on stderr). No external dependencies.\n\"\"\"\nimport argparse\nimport json\nimport os\nimport sys\n\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nfrom joint_angles import analyze_session  # noqa: E402\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    p.add_argument(\"--input\", required=True, help=\"path to session JSON\")\n    p.add_argument(\"--visibility-min\", type=float)\n    p.add_argument(\"--segment-tolerance\", type=float)\n    p.add_argument(\"--measurement-tolerance-deg\", type=float)\n    args = p.parse_args()\n    overrides = {k: v for k, v in {\n        \"visibility_min\": args.visibility_min,\n        \"segment_tolerance\": args.segment_tolerance,\n        \"measurement_tolerance_deg\": args.measurement_tolerance_deg,\n    }.items() if v is not None}\n    try:\n        with open(args.input, encoding=\"utf-8\") as fh:\n            session = json.load(fh)\n        if not isinstance(session, dict):\n            raise ValueError(\"session JSON must be an object.\")\n        result = analyze_session(session, overrides)\n        print(json.dumps(result, indent=2, allow_nan=False))\n    except FileNotFoundError:\n        print(json.dumps({\"error\": f\"input file not found: {args.input}\"}), file=sys.stderr)\n        return 2\n    except json.JSONDecodeError as e:\n        print(json.dumps({\"error\": f\"input is not valid JSON: {e}\"}), file=sys.stderr)\n        return 2\n    except (ValueError, OverflowError, ZeroDivisionError) as e:\n        print(json.dumps({\"error\": str(e)}), file=sys.stderr)\n        return 2\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n",
 "skills/knee-rom-scripted/scripts/joint_angles.py": "\"\"\"Knee range-of-motion (ROM) screen from 2D pose landmarks. Standard library only.\n\nTool contract adapted from the HW5 car-setup MCP tools (shanexf/car-setup-mcp):\na TOOL_SPEC with purpose, typed inputs, units and valid ranges, fixed range\nchecks that raise ValueError on invalid input, a structured result dict, a\nqualitative label, and a list of warnings near model limits.\n\nUnits: pixels for landmark coordinates, degrees for angles, seconds for time,\n0-10 numeric rating for pain. Clinical convention: knee flexion = 0 deg at\nfull extension, increasing as the knee bends.\n\"\"\"\nimport math\n\nTOOL_SPEC = {\n    \"name\": \"knee_rom_screen\",\n    \"purpose\": (\"Compute knee flexion per frame from hip-knee-ankle landmarks of a \"\n                \"side-view (sagittal) video, reject unreliable frames, summarize \"\n                \"extension lag and ROM, compare with clinician targets and the last \"\n                \"session, and return on_track / below_target / clinician_review.\"),\n    \"inputs\": {\n        \"side\": \"str, 'left' or 'right'\",\n        \"camera_view\": \"str, must be 'sagittal' for a valid 2D knee-flexion screen\",\n        \"coordinate_units\": \"str, 'pixels' or 'normalized' (normalized needs image size)\",\n        \"image_width_px, image_height_px\": \"int, 100-10000 px (needed for normalized)\",\n        \"frames\": \"list of {t_s, hip:[x,y,vis], knee:[x,y,vis], ankle:[x,y,vis]}, 3-10000 frames\",\n        \"targets.max_extension_lag_deg\": \"float, 0-30 deg (clinician-set)\",\n        \"targets.min_rom_deg\": \"float, 10-150 deg (clinician-set)\",\n        \"previous_session\": \"optional {extension_lag_deg, rom_deg, pain_0_10}\",\n        \"pain_0_10\": \"float, 0-10 (patient-reported)\",\n    },\n    \"assumptions\": [\n        \"2D image-plane angle; leg moves in a plane roughly parallel to the camera\",\n        \"unsigned included angle: hyperextension (negative flexion) cannot be detected\",\n        \"landmark visibility is a model confidence score, not a calibrated probability\",\n        \"thresholds are team-chosen screening defaults, not clinical standards\",\n        \"session extremes over the cleaned series; no per-repetition segmentation\",\n    ],\n}\n\nDEFAULT_SETTINGS = {\n    \"visibility_min\": 0.5,             # drop frame if any of hip/knee/ankle below this\n    \"segment_tolerance\": 0.15,         # drop frame if thigh/shank length deviates >15% from median\n    \"measurement_tolerance_deg\": 5.0,  # changes/misses smaller than this are within 2D pose error\n    \"min_valid_frames\": 5,\n    \"min_valid_fraction\": 0.6,\n    \"max_plausible_flexion_deg\": 150.0,\n    \"pain_review_level\": 6.0,          # pain >= this -> clinician review\n    \"pain_increase_review\": 2.0,       # pain increase >= this vs last session -> review\n    \"smoothing_min_frames\": 15,        # median-5 smoothing only for real video-length series\n}\n\nRANGES = {\n    \"image_width_px\": (100, 10000), \"image_height_px\": (100, 10000),\n    \"max_extension_lag_deg\": (0, 30), \"min_rom_deg\": (10, 150),\n    \"pain_0_10\": (0, 10), \"extension_lag_deg\": (0, 90), \"rom_deg\": (0, 160),\n    \"visibility_min\": (0.0, 1.0), \"segment_tolerance\": (0.01, 0.5),\n    \"measurement_tolerance_deg\": (0.0, 20.0), \"min_valid_frames\": (3, 1000),\n    \"min_valid_fraction\": (0.1, 1.0), \"max_plausible_flexion_deg\": (90, 180),\n    \"pain_review_level\": (0, 10), \"pain_increase_review\": (0, 10),\n    \"smoothing_min_frames\": (5, 100000),\n}\n\n\ndef _num(name, value):\n    if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):\n        raise ValueError(f\"{name} must be a finite number, got {value!r}.\")\n    if name in RANGES:\n        lo, hi = RANGES[name]\n        if not lo <= value <= hi:\n            raise ValueError(f\"{name}={value} is outside the valid range [{lo}, {hi}].\")\n    return float(value)\n\n\ndef included_angle_deg(a, b, c):\n    \"\"\"Angle ABC in degrees between vectors B->A and B->C (0-180).\"\"\"\n    bax, bay = a[0] - b[0], a[1] - b[1]\n    bcx, bcy = c[0] - b[0], c[1] - b[1]\n    na, nc = math.hypot(bax, bay), math.hypot(bcx, bcy)\n    if na == 0 or nc == 0:\n        raise ValueError(\"Coincident landmarks: a segment has zero length.\")\n    cosv = (bax * bcx + bay * bcy) / (na * nc)\n    return math.degrees(math.acos(max(-1.0, min(1.0, cosv))))\n\n\ndef knee_flexion_deg(hip, knee, ankle):\n    \"\"\"Clinical knee flexion: 180 - included hip-knee-ankle angle (0 = straight).\"\"\"\n    return 180.0 - included_angle_deg(hip, knee, ankle)\n\n\ndef _median(values):\n    s = sorted(values)\n    n = len(s)\n    return s[n // 2] if n % 2 else 0.5 * (s[n // 2 - 1] + s[n // 2])\n\n\ndef _classify_change(delta_better, tol):\n    \"\"\"delta_better > 0 means improvement.\"\"\"\n    if delta_better > tol:\n        return \"improved\"\n    if delta_better < -tol:\n        return \"worsened\"\n    return \"no_detectable_change\"\n\n\ndef analyze_session(session, settings=None):\n    cfg = dict(DEFAULT_SETTINGS)\n    cfg.update(settings or {})\n    for k, v in cfg.items():\n        cfg[k] = _num(k, v)\n    warnings, review = [], []\n\n    side = session.get(\"side\")\n    if side not in (\"left\", \"right\"):\n        raise ValueError(\"side must be 'left' or 'right'.\")\n    view = session.get(\"camera_view\")\n    if view not in (\"sagittal\", \"frontal\", \"oblique\", \"unknown\"):\n        raise ValueError(\"camera_view must be sagittal, frontal, oblique, or unknown.\")\n    units = session.get(\"coordinate_units\")\n    if units not in (\"pixels\", \"normalized\"):\n        raise ValueError(\"coordinate_units must be 'pixels' or 'normalized'.\")\n    sx = sy = 1.0\n    if units == \"normalized\":\n        if \"image_width_px\" not in session or \"image_height_px\" not in session:\n            raise ValueError(\"Normalized coordinates require image_width_px and image_height_px; \"\n                             \"angles on unscaled normalized x,y are distorted for non-square images.\")\n        sx = _num(\"image_width_px\", session[\"image_width_px\"])\n        sy = _num(\"image_height_px\", session[\"image_height_px\"])\n\n    targets = session.get(\"targets\")\n    if not isinstance(targets, dict):\n        raise ValueError(\"targets with max_extension_lag_deg and min_rom_deg are required (clinician-set).\")\n    lag_target = _num(\"max_extension_lag_deg\", targets.get(\"max_extension_lag_deg\"))\n    rom_target = _num(\"min_rom_deg\", targets.get(\"min_rom_deg\"))\n    if \"pain_0_10\" not in session:\n        raise ValueError(\"pain_0_10 (patient-reported, 0-10) is required.\")\n    pain = _num(\"pain_0_10\", session[\"pain_0_10\"])\n    prev = session.get(\"previous_session\")\n    if prev is not None:\n        prev = {k: _num(k, prev[k]) for k in (\"extension_lag_deg\", \"rom_deg\", \"pain_0_10\") if k in prev}\n\n    frames = session.get(\"frames\")\n    if not isinstance(frames, list) or not 3 <= len(frames) <= 10000:\n        raise ValueError(\"frames must be a list of 3-10000 frames.\")\n\n    rows, last_t = [], None\n    for i, f in enumerate(frames):\n        try:\n            t = _num(f\"frames[{i}].t_s\", f[\"t_s\"])\n            pts = {}\n            for name in (\"hip\", \"knee\", \"ankle\"):\n                x, y, vis = f[name]\n                x, y, vis = (_num(f\"frames[{i}].{name}\", v) for v in (x, y, vis))\n                if not 0 <= vis <= 1:\n                    raise ValueError(f\"frames[{i}].{name} visibility must be in [0, 1].\")\n                pts[name] = (x * sx, y * sy, vis)\n        except (KeyError, TypeError) as e:\n            raise ValueError(f\"frames[{i}] must have t_s and hip/knee/ankle as [x, y, visibility].\") from e\n        if last_t is not None and t <= last_t:\n            raise ValueError(f\"frames[{i}].t_s must be strictly increasing.\")\n        last_t = t\n        h, k, a = pts[\"hip\"], pts[\"knee\"], pts[\"ankle\"]\n        rows.append({\n            \"index\": i, \"t_s\": t,\n            \"flexion_deg\": knee_flexion_deg(h, k, a),\n            \"thigh_px\": math.hypot(h[0] - k[0], h[1] - k[1]),\n            \"shank_px\": math.hypot(a[0] - k[0], a[1] - k[1]),\n            \"min_visibility\": min(h[2], k[2], a[2]),\n            \"status\": \"valid\", \"reason\": None,\n        })\n\n    for r in rows:\n        if r[\"min_visibility\"] < cfg[\"visibility_min\"]:\n            r[\"status\"], r[\"reason\"] = \"rejected\", f\"low visibility ({r['min_visibility']:.2f})\"\n    vis_ok = [r for r in rows if r[\"status\"] == \"valid\"]\n    if vis_ok:\n        med_thigh = _median([r[\"thigh_px\"] for r in vis_ok])\n        med_shank = _median([r[\"shank_px\"] for r in vis_ok])\n        for r in vis_ok:\n            dt = r[\"thigh_px\"] / med_thigh - 1\n            ds = r[\"shank_px\"] / med_shank - 1\n            if max(abs(dt), abs(ds)) > cfg[\"segment_tolerance\"]:\n                r[\"status\"] = \"rejected\"\n                r[\"reason\"] = (f\"segment length off median (thigh {dt:+.0%}, shank {ds:+.0%}): \"\n                               \"out-of-plane motion or landmark error\")\n    else:\n        med_thigh = med_shank = None\n\n    valid = [r for r in rows if r[\"status\"] == \"valid\"]\n    n, nv = len(rows), len(valid)\n    frac = nv / n\n    if nv < cfg[\"min_valid_frames\"] or frac < cfg[\"min_valid_fraction\"]:\n        review.append(f\"insufficient reliable frames: {nv}/{n} valid \"\n                      f\"(need >= {int(cfg['min_valid_frames'])} and >= {cfg['min_valid_fraction']:.0%})\")\n    elif nv < n:\n        warnings.append(f\"{n - nv} of {n} frames rejected; see per-frame reasons\")\n    if view != \"sagittal\":\n        review.append(f\"camera_view is '{view}': 2D knee flexion is only valid from a side (sagittal) view\")\n\n    series = [r[\"flexion_deg\"] for r in valid]\n    smoothing = \"none\"\n    if nv >= cfg[\"smoothing_min_frames\"]:\n        sm = [_median(series[max(0, j - 2): j + 3]) for j in range(nv)]\n        for r, v in zip(valid, sm):\n            r[\"flexion_smoothed_deg\"] = v\n        series, smoothing = sm, \"centered median, window 5\"\n\n    summary = None\n    decision = \"clinician_review\"\n    target_status, change = {}, {}\n    if nv >= 2:\n        ext_lag = min(series)\n        peak_flex = max(series)\n        rom = peak_flex - ext_lag\n        summary = {\"extension_lag_deg\": ext_lag, \"peak_flexion_deg\": peak_flex, \"rom_deg\": rom,\n                   \"valid_frames\": nv, \"total_frames\": n, \"valid_fraction\": frac,\n                   \"median_thigh_px\": med_thigh, \"median_shank_px\": med_shank,\n                   \"smoothing\": smoothing}\n        if peak_flex > cfg[\"max_plausible_flexion_deg\"]:\n            review.append(f\"implausible peak flexion {peak_flex:.1f} deg for this exercise\")\n        if ext_lag < 2.0:\n            warnings.append(\"extension near 0 deg: hyperextension cannot be measured by an unsigned 2D angle\")\n        tol = cfg[\"measurement_tolerance_deg\"]\n\n        def status(miss):\n            return \"met\" if miss <= 0 else (\"within_tolerance\" if miss <= tol else \"not_met\")\n        target_status = {\n            \"extension_lag\": {\"value_deg\": ext_lag, \"target_max_deg\": lag_target,\n                              \"miss_deg\": ext_lag - lag_target, \"status\": status(ext_lag - lag_target)},\n            \"rom\": {\"value_deg\": rom, \"target_min_deg\": rom_target,\n                    \"miss_deg\": rom_target - rom, \"status\": status(rom_target - rom)},\n        }\n        if prev:\n            if \"extension_lag_deg\" in prev:\n                d = prev[\"extension_lag_deg\"] - ext_lag\n                change[\"extension_lag\"] = {\"previous_deg\": prev[\"extension_lag_deg\"], \"now_deg\": ext_lag,\n                                           \"improvement_deg\": d, \"class\": _classify_change(d, tol)}\n            if \"rom_deg\" in prev:\n                d = rom - prev[\"rom_deg\"]\n                change[\"rom\"] = {\"previous_deg\": prev[\"rom_deg\"], \"now_deg\": rom,\n                                 \"improvement_deg\": d, \"class\": _classify_change(d, tol)}\n            for key, c in change.items():\n                if c[\"class\"] == \"worsened\":\n                    review.append(f\"{key} worsened by {-c['improvement_deg']:.1f} deg vs last session \"\n                                  f\"(beyond {tol:g} deg measurement tolerance)\")\n        else:\n            warnings.append(\"no previous session supplied: trend cannot be assessed\")\n    else:\n        review.append(\"fewer than 2 valid frames: no angle summary\")\n\n    if pain >= cfg[\"pain_review_level\"]:\n        review.append(f\"pain {pain:g}/10 at or above review level {cfg['pain_review_level']:g}\")\n    if prev and \"pain_0_10\" in prev and pain - prev[\"pain_0_10\"] >= cfg[\"pain_increase_review\"]:\n        review.append(f\"pain increased from {prev['pain_0_10']:g} to {pain:g}\")\n\n    if not review and target_status:\n        statuses = [v[\"status\"] for v in target_status.values()]\n        decision = \"below_target\" if \"not_met\" in statuses else \"on_track\"\n        if \"within_tolerance\" in statuses:\n            warnings.append(\"a target is missed by less than the measurement tolerance: \"\n                            \"confirm with a goniometer before calling it met\")\n\n    return {\n        \"tool\": TOOL_SPEC[\"name\"],\n        \"patient_id\": session.get(\"patient_id\"),\n        \"exercise\": session.get(\"exercise\"),\n        \"side\": side, \"camera_view\": view, \"coordinate_units\": units,\n        \"settings\": cfg,\n        \"frames\": rows,\n        \"summary\": summary,\n        \"targets\": target_status,\n        \"change_vs_previous\": change,\n        \"pain_0_10\": pain,\n        \"decision\": decision,\n        \"review_triggers\": review,\n        \"warnings\": warnings,\n        \"assumptions\": TOOL_SPEC[\"assumptions\"],\n        \"scope\": (\"Screening aid only. Not a diagnosis. A clinician decides any change \"\n                  \"to the exercise plan.\"),\n    }\n"
}

SESSION_INPUT = "{\n \"patient_id\": \"P-anon-01\",\n \"exercise\": \"seated_knee_extension\",\n \"side\": \"right\",\n \"camera_view\": \"sagittal\",\n \"coordinate_units\": \"pixels\",\n \"image_width_px\": 1080,\n \"image_height_px\": 1920,\n \"frames\": [\n  {\n   \"t_s\": 0.0,\n   \"hip\": [\n    379,\n    1009,\n    0.98\n   ],\n   \"knee\": [\n    800,\n    1038,\n    0.96\n   ],\n   \"ankle\": [\n    785,\n    1439,\n    0.97\n   ]\n  },\n  {\n   \"t_s\": 0.5,\n   \"hip\": [\n    378,\n    1010,\n    0.98\n   ],\n   \"knee\": [\n    797,\n    1039,\n    0.96\n   ],\n   \"ankle\": [\n    947,\n    1409,\n    0.97\n   ]\n  },\n  {\n   \"t_s\": 1.0,\n   \"hip\": [\n    380,\n    1011,\n    0.98\n   ],\n   \"knee\": [\n    797,\n    1038,\n    0.96\n   ],\n   \"ankle\": [\n    1101,\n    1304,\n    0.97\n   ]\n  },\n  {\n   \"t_s\": 1.5,\n   \"hip\": [\n    380,\n    1010,\n    0.98\n   ],\n   \"knee\": [\n    801,\n    1037,\n    0.96\n   ],\n   \"ankle\": [\n    1179,\n    1169,\n    0.97\n   ]\n  },\n  {\n   \"t_s\": 2.0,\n   \"hip\": [\n    379,\n    1008,\n    0.98\n   ],\n   \"knee\": [\n    798,\n    1041,\n    0.96\n   ],\n   \"ankle\": [\n    1189,\n    1123,\n    0.97\n   ]\n  },\n  {\n   \"t_s\": 2.5,\n   \"hip\": [\n    381,\n    1009,\n    0.98\n   ],\n   \"knee\": [\n    799,\n    1038,\n    0.96\n   ],\n   \"ankle\": [\n    1076,\n    1067,\n    0.97\n   ]\n  },\n  {\n   \"t_s\": 3.0,\n   \"hip\": [\n    381,\n    1010,\n    0.98\n   ],\n   \"knee\": [\n    798,\n    1040,\n    0.96\n   ],\n   \"ankle\": [\n    1130,\n    1262,\n    0.97\n   ]\n  },\n  {\n   \"t_s\": 3.5,\n   \"hip\": [\n    381,\n    1011,\n    0.98\n   ],\n   \"knee\": [\n    798,\n    1040,\n    0.96\n   ],\n   \"ankle\": [\n    955,\n    1409,\n    0.97\n   ]\n  },\n  {\n   \"t_s\": 4.0,\n   \"hip\": [\n    381,\n    1009,\n    0.98\n   ],\n   \"knee\": [\n    801,\n    1038,\n    0.96\n   ],\n   \"ankle\": [\n    722,\n    1433,\n    0.32\n   ]\n  }\n ],\n \"targets\": {\n  \"max_extension_lag_deg\": 5,\n  \"min_rom_deg\": 90\n },\n \"previous_session\": {\n  \"extension_lag_deg\": 12,\n  \"rom_deg\": 70,\n  \"pain_0_10\": 3\n },\n \"pain_0_10\": 3\n}"

for relative, content in SKILL_FILES.items():
    relative = Path(relative)
    name = relative.parts[1]
    run_dir = MARKDOWN_RUN if name == 'knee-rom-markdown' else ASSISTED_RUN
    target = run_dir / '.agents' / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding='utf-8')
    print(target.relative_to(ROOT))
for run_dir in [MARKDOWN_RUN, ASSISTED_RUN]:
    (run_dir / 'session_input.json').write_text(SESSION_INPUT, encoding='utf-8')
    print((run_dir / 'session_input.json').relative_to(ROOT))

PROBLEM = """A rehab patient performed a seated right-knee extension, filmed from the
side with a phone. The session data (pose landmark frames, clinician targets,
last session, and pain rating) is in session_input.json in the current folder;
read it. Screen the knee range of motion: decide which frames to trust,
summarize extension lag, peak flexion, and ROM, compare with the targets and
the last session, and give the decision with your reasoning."""

def opencode_config(run_dir, scripted):
    rules = [
        {'action':'*','resource':'*','effect':'deny'},
        {'action':'skill','resource':'knee-rom-*','effect':'allow'},
        {'action':'read','resource':'*','effect':'allow'},
        {'action':'shell','resource':'*','effect':'allow' if scripted else 'deny'},
    ]
    return {
        '$schema': 'https://opencode.ai/config.json',
        'model': 'local/qwen', 'default_agent': 'knee-demo',
        'skills': [str(run_dir / '.agents/skills')],
        'providers': {'local': {
            'name': 'Local Hugging Face Transformers',
            'package': '@opencode/ai/providers/openai-compatible',
            'settings': {'baseURL':'http://127.0.0.1:8000/v1'},
            'compatibility': {'streamUsage':False,'maxTokensField':'max_tokens','requireFinishReason':False},
            'models': {'qwen': {'modelID':MODEL_ID,
                'capabilities':{'tools':True,'input':['text'],'output':['text']},
                'limit':{'context':32768,'output':4000},
                'body':{'temperature':0.2,'top_p':0.8}}}}},
        'agents': {'knee-demo': {
            'description':'Bounded rehab joint-angle screening comparison',
            'mode':'primary', 'steps':8,
            'system':('You are a rehabilitation biomechanics teaching assistant. The skill tool loads '
                      'instructions; the skill name is NOT itself an executable tool. Load the requested '
                      'skill exactly once, then solve the task by following those instructions. Use only '
                      'native skill, read, and shell tools when allowed. Return the final tables and '
                      'decision directly as ordinary text, not as a tool call.'),
            'permissions': rules}},
        'permissions': rules,
    }
for run_dir in [MARKDOWN_RUN, ASSISTED_RUN]:
    (run_dir/'opencode.jsonc').write_text(json.dumps(opencode_config(run_dir, run_dir==ASSISTED_RUN), indent=2))
print('Both agents are ready. Only Case 2 can execute Python.')

import html
from IPython.display import HTML, Markdown, display

def run_skill(label, folder, skill):
    prompt = (f'Load the native skill {skill} once, then follow it. '
              + PROBLEM + ' Give a frame table, a summary table, and the decision as ordinary text.')
    started = time.monotonic()
    print(label.upper(), flush=True)
    print('MODEL:', MODEL_ID, flush=True)
    print('PROMPT:\n' + prompt, flush=True)
    transcript = TRANSCRIPTS / f'{label}.jsonl'
    with transcript.open('w') as saved:
        process = subprocess.Popen(
            ['opencode','run','--standalone','--auto','--archived','--format','json',
             '--thinking','--agent','knee-demo','--model','local/qwen',prompt],
            cwd=folder,
            # OpenCode resolves relative paths from $PWD, not the process cwd: set both.
            env={**os.environ,'OPENCODE_CONFIG':str(folder/'opencode.jsonc'),'PWD':str(folder)},
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                saved.write(line)
                saved.flush()
                try:
                    event = json.loads(line)
                except json.JSONDecodeError:
                    print(line.rstrip(), flush=True)
                    continue
                kind = event.get('type')
                part = event.get('part', {})
                print(f'\n[{time.monotonic()-started:.1f}s] EVENT: {kind}', flush=True)
                if kind == 'step_start':
                    print('Model generating a response…', flush=True)
                if kind == 'tool_use':
                    state = part.get('state', {})
                    print('TOOL:', part.get('tool'), state.get('status'), flush=True)
                    print('INPUT:\n' + json.dumps(state.get('input', {}), indent=2), flush=True)
                    if state.get('output'):
                        print('OUTPUT:\n' + str(state['output']), flush=True)
                    if state.get('error'):
                        print('ERROR:', state['error'], flush=True)
                elif kind in ('reasoning', 'thinking'):
                    print(part.get('text', ''), flush=True)
                elif kind == 'text':
                    print(part.get('text', ''), flush=True)   # plain text survives in saved outputs
                    display(Markdown(part.get('text', '')))
                elif kind == 'step_finish':
                    print('Stop reason:', part.get('reason'), flush=True)
                    print('Tokens:', part.get('tokens', {}), flush=True)
                elif kind == 'error':
                    print(event.get('error', {}), flush=True)
                payload = html.escape(json.dumps(event, indent=2, ensure_ascii=False))
                display(HTML(f'<details><summary>Raw {html.escape(str(kind))} event</summary><pre>{payload}</pre></details>'))
            process.wait()
        except KeyboardInterrupt:
            process.terminate()
            raise
    print(f'Finished in {time.monotonic()-started:.1f}s; process exit: {process.returncode}', flush=True)
    print('Raw trace saved:', transcript, flush=True)
    return label, transcript

### Optional: check the script before the agent runs
Runs the bundled script directly (no agent) once on the session file and once
on an invalid copy, so the expected output and the failure behavior are on record.

In [ ]:
script = ASSISTED_RUN / '.agents/skills/knee-rom-scripted/scripts/analyze_knee_rom.py'
ok = subprocess.run(['python', str(script), '--input', str(ASSISTED_RUN/'session_input.json')],
                    capture_output=True, text=True)
result = json.loads(ok.stdout)
print('exit', ok.returncode, '| decision:', result['decision'])
for f in result['frames']:
    print(f"{f['t_s']:>4}s  flexion {f['flexion_deg']:6.2f}°  thigh {f['thigh_px']:6.1f}px  "
          f"shank {f['shank_px']:6.1f}px  {f['status']:8s} {f['reason'] or ''}")
print(json.dumps({k: result[k] for k in ('summary','targets','change_vs_previous','review_triggers','warnings')}, indent=2))

bad = json.loads(SESSION_INPUT); del bad['targets']
bad_path = ROOT / 'invalid_session.json'; bad_path.write_text(json.dumps(bad))
err = subprocess.run(['python', str(script), '--input', str(bad_path)], capture_output=True, text=True)
print('\nInvalid input -> exit', err.returncode, err.stderr.strip())

## 3. Start the local model
Run once and wait for **Model ready**. Leave it running for both cases.

In [ ]:
!transformers serve {MODEL_ID} --device cuda:0 --dtype float16 \
    --attn-implementation sdpa --reasoning off > /content/knee-model.log 2>&1 &
!until curl -sf http://127.0.0.1:8000/health > /dev/null; do sleep 2; done
print("Model ready:", MODEL_ID)

## 4. Case 1: Markdown only
Fresh OpenCode session. Watch for the `skill` load, the `read` of the session file,
and the hand calculations in the answer. Shell is denied.

In [ ]:
markdown_run = run_skill('markdown-only', MARKDOWN_RUN, 'knee-rom-markdown')

## 5. Case 2: Script assisted
Fresh OpenCode session. Watch for the `skill` load, the exact `shell` command,
the script's JSON output, and the agent's interpretation.

In [ ]:
scripted_run = run_skill('script-assisted', ASSISTED_RUN, 'knee-rom-scripted')

## 6. Independent check data
Prints the raw coordinates of one frame for a hand calculation (written up
separately), plus a second implementation with NumPy that does not import the
skill's code. The NumPy version is a cross-check of the arithmetic only; it is
not the human independent check.

In [ ]:
import numpy as np
s = json.loads(SESSION_INPUT)
f0 = s['frames'][0]
print('Frame t=0.0 s  hip', f0['hip'][:2], ' knee', f0['knee'][:2], ' ankle', f0['ankle'][:2])
for f in s['frames']:
    H, K, A = (np.array(f[k][:2], float) for k in ('hip','knee','ankle'))
    u, v = H - K, A - K
    theta = np.degrees(np.arccos(np.clip(u @ v / (np.linalg.norm(u)*np.linalg.norm(v)), -1, 1)))
    print(f"t={f['t_s']:>3}s  flexion={180-theta:6.2f}°  |thigh|={np.linalg.norm(u):6.1f}  "
          f"|shank|={np.linalg.norm(v):6.1f}  min vis={min(f[k][2] for k in ('hip','knee','ankle')):.2f}")

## Discuss the comparison
Compare the two responses: which frames each agent trusted, whether the flexion
convention and arithmetic were right, how each handled the measurement
tolerance, the targets and the last session, which review triggers and
omissions were mentioned, and what changed when the agent could run the
bundled script.